In [10]:
import requests
import pandas as pd
import time

wikipedia = pd.read_csv("../data/raw/wikipedia.csv")
page_ids = wikipedia['page_id'].tolist()

# Resume from previous run
try:
    existing = pd.read_csv("creation_dates.csv")
    creation_dates = dict(zip(existing['page_id'], existing['creation_date']))
    print(f"Loaded {len(creation_dates)} existing creation dates, resuming...")
    page_ids = [pid for pid in page_ids if pid not in creation_dates]
    print(f"{len(page_ids)} remaining")
except FileNotFoundError:
    creation_dates = {}

headers = {'User-Agent': 'WikiQualityResearch/1.0 (academic project)'}
session = requests.Session()
session.headers.update(headers)

start_time = time.time()

for i, pid in enumerate(page_ids):
    params = {
        'action': 'query',
        'prop': 'revisions',
        'rvlimit': '1',
        'rvdir': 'newer',
        'rvprop': 'timestamp',
        'pageids': str(pid),
        'format': 'json',
    }
    try:
        for attempt in range(5):
            resp = session.get('https://en.wikipedia.org/w/api.php', params=params, timeout=30)
            if resp.status_code == 429:
                wait = int(resp.headers.get('Retry-After', 5))
                print(f"Rate limited at {i+1}/{len(page_ids)}, waiting {wait}s...")
                time.sleep(wait)
                continue
            elif resp.status_code != 200:
                print(f"ABORT: HTTP {resp.status_code} at page_id={pid}")
                break
            else:
                break
        else:
            print(f"ABORT: Still rate limited after 5 retries at page_id={pid}")
            break
        if resp.status_code != 200:
            break
        data = resp.json()
        if 'error' in data:
            print(f"ABORT: API error at page_id={pid}: {data['error']}")
            break
        pages = data.get('query', {}).get('pages', {})
        for page_id, info in pages.items():
            revs = info.get('revisions', [])
            if revs:
                creation_dates[int(page_id)] = revs[0]['timestamp']
            else:
                print(f"WARNING: No revisions for page_id={pid}, skipping")
    except Exception as e:
        print(f"ABORT: Exception at page_id={pid}: {e}")
        break

    time.sleep(0.5)

    if (i + 1) % 500 == 0:
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed
        remaining = (len(page_ids) - i - 1) / rate
        print(f"{i+1}/{len(page_ids)} | {len(creation_dates)} fetched | {elapsed:.0f}s elapsed | ~{remaining:.0f}s remaining")

elapsed = time.time() - start_time
print(f"\nDone in {elapsed:.0f}s. Got {len(creation_dates)}/{len(page_ids)} creation dates.")

result = pd.DataFrame([{'page_id': pid, 'creation_date': ts} for pid, ts in creation_dates.items()])
result.to_csv("creation_dates.csv", index=False)
print("Saved to creation_dates.csv")

Loaded 8529 existing creation dates, resuming...
6633 remaining
500/6633 | 9029 fetched | 280s elapsed | ~3433s remaining
1000/6633 | 9529 fetched | 560s elapsed | ~3156s remaining
1500/6633 | 10029 fetched | 841s elapsed | ~2878s remaining
2000/6633 | 10529 fetched | 1120s elapsed | ~2594s remaining
2500/6633 | 11029 fetched | 1399s elapsed | ~2312s remaining
3000/6633 | 11529 fetched | 1677s elapsed | ~2030s remaining
3500/6633 | 12029 fetched | 1957s elapsed | ~1752s remaining
4000/6633 | 12529 fetched | 2236s elapsed | ~1472s remaining
4500/6633 | 13029 fetched | 2515s elapsed | ~1192s remaining
5000/6633 | 13529 fetched | 2796s elapsed | ~913s remaining
5500/6633 | 14029 fetched | 3076s elapsed | ~634s remaining
6000/6633 | 14174 fetched | 3356s elapsed | ~354s remaining
6500/6633 | 14254 fetched | 3635s elapsed | ~74s remaining

Done in 3709s. Got 14275/6633 creation dates.
Saved to creation_dates.csv
